<a href="https://colab.research.google.com/github/CathalD/BlueCarbon_Workflow_V1.0/blob/main/CovariateExtractionWorkflow.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║   COASTAL BLUE CARBON — COVARIATE EXTRACTION  v4.0                      ║
# ║   Canada · Tidal Marsh + Seagrass · VM0033                              ║
# ║                                                                          ║
# ║   Run cells top to bottom in order.                                      ║
# ║   The only cells you need to edit are:                                   ║
# ║     CELL 2  — your project settings & date ranges                        ║
# ║     CELL 3  — your AOI (draw, upload, or paste an asset path)            ║
# ║     CELL 4  — upload your core profiles CSV                              ║
# ╚══════════════════════════════════════════════════════════════════════════╝


# ──────────────────────────────────────────────────────────────────────────
# CELL 1 ── Install packages & authenticate  (run once per session)
# ──────────────────────────────────────────────────────────────────────────

import subprocess, sys

print("Installing packages…")
for pkg in ["earthengine-api", "geemap", "geopandas"]:
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

import ee
import geemap
import pandas as pd
import numpy as np
from google.colab import files
from tqdm.notebook import tqdm
from datetime import datetime
from IPython.display import display, clear_output
import io, os, zipfile, tempfile, traceback

ee.Authenticate()
print("\n✅ Cell 1 done — packages installed, GEE authenticated.")


Installing packages…

✅ Cell 1 done — packages installed, GEE authenticated.


In [3]:
# ──────────────────────────────────────────────────────────────────────────
# CELL 2 ── Configuration  ← EDIT THIS CELL
# ──────────────────────────────────────────────────────────────────────────

# ── GEE project (required) ────────────────────────────────────────────────
GEE_PROJECT    = "northstarlabs"

# ── Output destinations ───────────────────────────────────────────────────
EXPORT_FOLDER  = "BlueCarbon_CoastalBC"   # Google Drive folder
PROJECT_YEAR   = "2020_2023"              # label used in all output filenames
EXPORT_CRS     = "EPSG:3347"             # Canada Albers Equal Area
EXPORT_SCALE   = 25                       # covariate raster resolution (m)
EMBED_SCALE    = 10                       # embedding raster resolution (m)

# ── To save the AOI boundary as a GEE Asset, paste your project ID ────────
# ── Leave as None to skip ─────────────────────────────────────────────────
ASSET_PROJECT  = "northstarlabs"

# ── AOI buffer ────────────────────────────────────────────────────────────
COV_BUFFER_M   = 500                      # metres buffer around AOI for exports

# ── Sentinel-2 date range ─────────────────────────────────────────────────
S2_START       = "2020-01-01"
S2_END         = "2023-12-31"

# ── Sentinel-1 SAR date range ─────────────────────────────────────────────
SAR_START      = "2020-01-01"
SAR_END        = "2023-12-31"

# ── TerraClimate date range (for MAT / MAP) ───────────────────────────────
TC_START       = "2000-01-01"
TC_END         = "2022-12-31"

# ── Climate analog filter ─────────────────────────────────────────────────
# Keeps only global cores whose climate is similar to your AOI.
# Set to False to skip and use all cores.
USE_CLIMATE_FILTER = True
MAT_TOLERANCE      = 8.0    # ± °C
MAP_TOLERANCE      = 0.60   # ± fraction (0.60 = ±60%)

# ── Similarity raster (transfer learning weights) ─────────────────────────
# Builds a hybrid similarity surface (embeddings + covariates) comparing
# each AOI pixel to the centroid of your training core profiles.
# The `similarity` band is used as case.weights in the downstream R model.
# Adds ~5 min. Set to False to skip.
USE_SIMILARITY = True

# ── Internal S2 extraction settings (rarely need changing) ────────────────
S2_MAX_CLOUD   = 20     # max cloud % per scene
S2_IMAGE_LIMIT = 20     # max scenes per batch area
S2_BUFFER_M    = 5000   # metres buffer around each extraction batch

# Initialise Earth Engine
ee.Initialize(project=GEE_PROJECT)

# ── Canonical 27-band list — must match R workflow column names exactly ────
CANONICAL_BANDS = [
    "elevation_m", "slope", "elevationRelMHW", "twi",
    "dist_to_channel_m", "tidal_flat_prob", "coastal_dist_m",
    "VV_mean", "VH_mean", "VVVH_ratio",
    "B", "G", "R", "B5", "B6", "B7", "NIR", "SWIR1", "SWIR2",
    "NDVI_median", "LSWI_median", "mNDWI_median",
    "NDVI_stdDev", "SAVI_median", "tidal_wetness",
    "MAT_C", "MAP_mm",
]
assert len(CANONICAL_BANDS) == 27, f"Expected 27 canonical bands, got {len(CANONICAL_BANDS)}"

print("✅ Cell 2 done — configuration loaded.")
print(f"   Project        : {GEE_PROJECT}")
print(f"   Outputs        : Drive/{EXPORT_FOLDER}  |  CRS: {EXPORT_CRS}  |  Scale: {EXPORT_SCALE} m")
print(f"   S2             : {S2_START} → {S2_END}")
print(f"   Climate filter : {USE_CLIMATE_FILTER}  |  Similarity raster : {USE_SIMILARITY}")


✅ Cell 2 done — configuration loaded.
   Project        : northstarlabs
   Outputs        : Drive/BlueCarbon_CoastalBC  |  CRS: EPSG:3347  |  Scale: 25 m
   S2             : 2020-01-01 → 2023-12-31
   Climate filter : True  |  Similarity raster : True


In [10]:
# ──────────────────────────────────────────────────────────────────────────
# CELL 3 ── Define your AOI  ← CHOOSE ONE METHOD, comment out the others
# ──────────────────────────────────────────────────────────────────────────

# ── METHOD A: Draw on an interactive map ──────────────────────────────────
# Run this cell, draw a polygon on the map using the toolbar shape tool,
# then uncomment the two "confirm" lines below and run again.

#Map = geemap.Map(center=[49.5, -124.5], zoom=8)
#Map.add_basemap("SATELLITE")
#print("Draw your AOI polygon using the toolbar shape/pentagon icon.")
#print("When done, uncomment the two lines below and re-run this cell.")
#display(Map)

# Uncomment these two lines after drawing:
#aoi = Map.draw_last_feature.geometry()
#print("✅ AOI set from drawn polygon.")

# ── METHOD B: Load from a GEE Asset ──────────────────────────────────────
aoi = ee.FeatureCollection('projects/blue-carbon-hub/assets/Charlies_Place_KBA_boundaryFile_2025').geometry()
print("✅ AOI loaded from GEE Asset.")

# ── METHOD C: Upload a GeoJSON or Shapefile (.zip) ────────────────────────
# uploaded = files.upload()
# fname    = list(uploaded.keys())[0]
# if fname.endswith(".zip"):
#     import geopandas as gpd
#     with tempfile.TemporaryDirectory() as tmp:
#         with zipfile.ZipFile(fname, "r") as z: z.extractall(tmp)
#         shp = [f for f in os.listdir(tmp) if f.endswith(".shp")][0]
#         gdf = gpd.read_file(os.path.join(tmp, shp))
#     aoi = geemap.geopandas_to_ee(gdf).geometry().union()
# else:
#     aoi = geemap.geojson_to_ee(fname).geometry().union()
# print(f"✅ AOI loaded from {fname}.")

aoi_cov = aoi.buffer(COV_BUFFER_M)
print(f"   AOI buffer: {COV_BUFFER_M} m")
print("✅ Cell 3 done — AOI defined.")


✅ AOI loaded from GEE Asset.
   AOI buffer: 500 m
✅ Cell 3 done — AOI defined.


In [11]:
# ──────────────────────────────────────────────────────────────────────────
# CELL 4 ── Load core profiles CSV  ← UPLOAD YOUR FILE HERE
# ──────────────────────────────────────────────────────────────────────────
# Upload the output of 05_filter_profiles.R
# Required columns: profile_id, latitude, longitude, dataset

print("Select your combined_profiles_filtered.csv when the dialog appears…")
uploaded    = files.upload()
fname       = list(uploaded.keys())[0]
df_profiles = pd.read_csv(io.BytesIO(list(uploaded.values())[0]))
df_profiles["profile_id"] = df_profiles["profile_id"].astype(str)

required = {"profile_id", "latitude", "longitude", "dataset"}
missing  = required - set(df_profiles.columns)
if missing:
    raise ValueError(f"Missing required columns: {missing}")

df_pts = df_profiles.dropna(subset=["latitude", "longitude"]).copy()

# ── Coordinate sanity check ───────────────────────────────────────────────
n_dropped = len(df_profiles) - len(df_pts)
if n_dropped:
    print(f"  ⚠ {n_dropped} rows dropped for missing lat/lon")

lat_ok = df_pts["latitude"].between(-90, 90).all()
lon_ok = df_pts["longitude"].between(-180, 180).all()
if not lat_ok:
    print(f"  ⚠ Latitude values out of range [-90, 90] — check your CSV")
if not lon_ok:
    print(f"  ⚠ Longitude values out of range [-180, 180] — check your CSV")

# ── Duplicate profile_id check ────────────────────────────────────────────
dupes = df_pts["profile_id"].duplicated().sum()
if dupes:
    print(f"  ⚠ {dupes} duplicate profile_id values detected — downstream merges may produce extra rows")

print(f"\n✅ Cell 4 done — {len(df_pts):,} profiles loaded.")
print(f"   Datasets : {df_pts['dataset'].value_counts().to_dict()}")
print(f"   Lat range: {df_pts['latitude'].min():.2f} → {df_pts['latitude'].max():.2f}")
print(f"   Lon range: {df_pts['longitude'].min():.2f} → {df_pts['longitude'].max():.2f}")
display(df_pts[["profile_id","dataset","latitude","longitude"]].head())



Select your combined_profiles_filtered.csv when the dialog appears…


Saving janousek_profiles.csv to janousek_profiles (1).csv

✅ Cell 4 done — 1,284 profiles loaded.
   Datasets : {'Janousek': 1284}
   Lat range: 15.03 → 59.77
   Lon range: -151.87 → -92.71


,profile_id,dataset,latitude,longitude
0,1,Janousek,38.2031,-122.0270
1,10,Janousek,38.2000,-122.0276
2,100,Janousek,44.6254,-124.0213
3,1002,Janousek,40.8023,-124.1798
4,1003,Janousek,40.8023,-124.1815


In [12]:
# ──────────────────────────────────────────────────────────────────────────
# CELL 5 ── Build GEE image stacks  (no editing needed)
# ──────────────────────────────────────────────────────────────────────────
#
# NOTE on S2 masking strategy:
#   Two separate S2 compositing functions are defined:
#
#   _s2_median_points(region)
#     Cloud mask ONLY — no tidal inundation mask.
#     Used for point extraction at core locations (Cell 8).
#     Core profiles are in tidal marsh / seagrass by definition — they are
#     wet environments. Applying an NDWI < 0.1 inundation mask removes
#     valid pixels at these sites and returns 100% NA for all S2 bands.
#
#   _s2_median_raster(region)
#     Cloud mask + tidal inundation mask (NDWI < 0.1).
#     Used only for the AOI covariate raster (Cell 8 post-merge build,
#     Cell 9 similarity, Cell 10 export). For raster products we want
#     exposed-surface reflectance only, so the inundation mask is correct.
#
#   _s2_raw_bands(s2_composite) / _s2_derived_bands(s2_composite)
#     Shared band extractor — called by BOTH paths above.
#     Guarantees identical band names and arithmetic across point and raster
#     outputs. Band names match CANONICAL_BANDS exactly.

print("Building image stacks…")

# ── Topography & Channels (7 bands) ──────────────────────────────────────
dem         = ee.Image("NASA/NASADEM_HGT/001").select("elevation")
elevation_m = dem.rename("elevation_m")
slope       = ee.Terrain.slope(dem).rename("slope")
elev_mhw    = dem.subtract(0.5).rename("elevationRelMHW")

slope_rad = ee.Terrain.slope(dem).multiply(3.14159 / 180)
tan_slope = slope_rad.tan().max(0.001)
contrib   = (dem.gte(-9999).unmask(0)
             .reduceNeighborhood(reducer=ee.Reducer.sum(),
                                  kernel=ee.Kernel.circle(**{"radius":20,"units":"pixels"}))
             .max(1))
twi = contrib.divide(tan_slope).log().rename("twi")

channel_mask = (ee.Image("JRC/GSW1_4/GlobalSurfaceWater")
                .select("occurrence").gt(30).unmask(0)
                .focalMax(30, "circle", "meters"))
dist_ch      = (channel_mask
                .fastDistanceTransform(500,"pixels","squared_euclidean")
                .sqrt().multiply(30).rename("dist_to_channel_m").float())

try:
    tidal_flat = (ee.ImageCollection("UQ/murray/Intertidal/v1_1/global_intertidal")
                  .filterBounds(ee.Geometry.BBox(-180,-90,180,90))
                  .mosaic().select("classification").eq(1).unmask(0)
                  .rename("tidal_flat_prob").float())
    print("  ✓ Murray intertidal loaded")
except Exception:
    tidal_flat = ee.Image(0).rename("tidal_flat_prob").float()
    print("  ⚠ Murray intertidal unavailable — tidal_flat_prob set to 0")

water_mask   = ee.Image("JRC/GSW1_4/GlobalSurfaceWater").select("occurrence").gt(50).unmask(0)
coastal_dist = (water_mask.fastDistanceTransform(500,"pixels","squared_euclidean")
                .sqrt().multiply(30).rename("coastal_dist_m").float())

stack_topo = (elevation_m.addBands(slope).addBands(elev_mhw).addBands(twi)
              .addBands(dist_ch).addBands(tidal_flat).addBands(coastal_dist))

# ── Validate topo band count ──────────────────────────────────────────────
topo_bands = stack_topo.bandNames().getInfo()
assert len(topo_bands) == 7, f"Expected 7 topo bands, got {len(topo_bands)}: {topo_bands}"
print(f"  ✓ Topography & Channels: {topo_bands}")

# ── Sentinel-1 SAR (3 bands) ──────────────────────────────────────────────
# VVVH_ratio = VV − VH in dB (equivalent to power ratio in linear space).
# S1 GRD backscatter is delivered in dB, so subtraction is physically correct.
# Do NOT use .divide() here — that would be dimensionally incorrect.
s1 = (ee.ImageCollection("COPERNICUS/S1_GRD")
      .filterDate(SAR_START, SAR_END)
      .filter(ee.Filter.listContains("transmitterReceiverPolarisation","VV"))
      .filter(ee.Filter.listContains("transmitterReceiverPolarisation","VH"))
      .filter(ee.Filter.eq("instrumentMode","IW"))
      .map(lambda img: img.updateMask(img.select("VV").gt(-30))))

n_s1 = s1.size().getInfo()
if n_s1 == 0:
    print("  ⚠ No S1 scenes found for the SAR date range — VV/VH bands will be empty")
else:
    print(f"  ✓ S1: {n_s1} scenes loaded")

s1_mean   = s1.mean()
stack_sar = (s1_mean.select("VV").rename("VV_mean")
             .addBands(s1_mean.select("VH").rename("VH_mean"))
             .addBands(s1_mean.select("VV").subtract(s1_mean.select("VH")).rename("VVVH_ratio")))

sar_bands = stack_sar.bandNames().getInfo()
assert len(sar_bands) == 3, f"Expected 3 SAR bands, got {len(sar_bands)}: {sar_bands}"
print(f"  ✓ Sentinel-1 SAR: {sar_bands}")

# ── TerraClimate MAT & MAP (2 bands) ─────────────────────────────────────
tc      = ee.ImageCollection("IDAHO_EPSCOR/TERRACLIMATE").filterDate(TC_START, TC_END)
tc_mean = tc.mean()
mat     = tc_mean.expression("((tmmn+tmmx)/2.0)/10.0",
                               {"tmmn":tc_mean.select("tmmn"),"tmmx":tc_mean.select("tmmx")}).rename("MAT_C")
stack_climate = mat.addBands(tc_mean.select("pr").multiply(12).rename("MAP_mm"))

clim_bands = stack_climate.bandNames().getInfo()
assert len(clim_bands) == 2, f"Expected 2 climate bands, got {len(clim_bands)}: {clim_bands}"
print(f"  ✓ TerraClimate: {clim_bands}")

# ── Google Satellite Embedding V1 (64 bands) ──────────────────────────────
stack_embed = ee.ImageCollection("GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL").median()
embed_bands = stack_embed.bandNames().getInfo()
print(f"  ✓ Satellite Embeddings V1: {len(embed_bands)} bands")

# ── S2 compositing functions ──────────────────────────────────────────────
def _mask_s2_points(image):
    """Cloud mask only — for point extraction at tidal wetland core locations."""
    qa    = image.select("QA60")
    cloud = qa.bitwiseAnd(1<<10).eq(0).And(qa.bitwiseAnd(1<<11).eq(0))
    return image.updateMask(cloud).divide(10000)

def _mask_s2_raster(image):
    """Cloud mask + tidal inundation mask — for AOI raster products only."""
    qa    = image.select("QA60")
    cloud = qa.bitwiseAnd(1<<10).eq(0).And(qa.bitwiseAnd(1<<11).eq(0))
    tide  = image.normalizedDifference(["B3","B8"]).lt(0.1)
    return image.updateMask(cloud.And(tide)).divide(10000)

def _s2_median_points(region):
    """
    S2 composite for point extraction — cloud mask only, no seasonal filter.

    calendarRange is intentionally excluded here. Core profiles span Canada
    and are not guaranteed to have growing-season coverage within any given
    batch region. Applying a May-Sept filter empties the collection at many
    locations (especially cloud-prone coastal BC) and returns 100% NA for all
    S2 bands. The seasonal filter is retained in _s2_median_raster, where we
    specifically want exposed-surface summer reflectance for the AOI raster.
    """
    return (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
            .filterDate(S2_START, S2_END).filterBounds(region)
            .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", S2_MAX_CLOUD))
            .limit(S2_IMAGE_LIMIT, "CLOUDY_PIXEL_PERCENTAGE")
            .map(_mask_s2_points)
            .median().clip(region))

def _s2_median_raster(region):
    """S2 composite for raster export — cloud + tidal inundation mask."""
    return (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
            .filterDate(S2_START, S2_END).filterBounds(region)
            .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", S2_MAX_CLOUD))
            .filter(ee.Filter.calendarRange(5, 9, "month"))
            .limit(S2_IMAGE_LIMIT, "CLOUDY_PIXEL_PERCENTAGE")
            .map(_mask_s2_raster)
            .median().clip(region))

# ── S2 band extractors (kept separate — called independently in extraction) ─
def _s2_raw_bands(s2_composite):
    """
    Extract 9 raw reflectance bands from a pre-built S2 composite.
    Band names match CANONICAL_BANDS exactly.
    Called for raw reflectance pass in point extraction (Cell 8)
    and for raster export via build_covariate_stack().
    """
    return (s2_composite.select("B2").rename("B")
            .addBands(s2_composite.select("B3").rename("G"))
            .addBands(s2_composite.select("B4").rename("R"))
            .addBands(s2_composite.select("B5").rename("B5"))
            .addBands(s2_composite.select("B6").rename("B6"))
            .addBands(s2_composite.select("B7").rename("B7"))
            .addBands(s2_composite.select("B8").rename("NIR"))
            .addBands(s2_composite.select("B11").rename("SWIR1"))
            .addBands(s2_composite.select("B12").rename("SWIR2")))

def _s2_derived_bands(s2_composite):
    """
    Compute 5 derived S2 indices from a pre-built composite.
    Band names match CANONICAL_BANDS exactly.
    Called for derived index pass in point extraction (Cell 8)
    and for raster export via build_covariate_stack().

    NOTE: All indices use the raw S2 band selectors (B2–B12), NOT the
    renamed bands from _s2_raw_bands(), so both functions can be called
    independently on the same composite without requiring the other.
    """
    return (
        s2_composite.normalizedDifference(["B8","B4"]).rename("NDVI_median")
        .addBands(s2_composite.normalizedDifference(["B8","B11"]).rename("LSWI_median"))
        .addBands(s2_composite.normalizedDifference(["B3","B11"]).rename("mNDWI_median"))
        .addBands(s2_composite.expression(
            "((NIR-R)/(NIR+R+0.5))*1.5",
            {"NIR": s2_composite.select("B8"), "R": s2_composite.select("B4")}
        ).rename("SAVI_median"))
        .addBands(s2_composite.expression(
            "0.1511*B+0.1973*G+0.3283*R+0.3407*NIR-0.7117*SW1-0.4559*SW2",
            {"B":   s2_composite.select("B2"),
             "G":   s2_composite.select("B3"),
             "R":   s2_composite.select("B4"),
             "NIR": s2_composite.select("B8"),
             "SW1": s2_composite.select("B11"),
             "SW2": s2_composite.select("B12")}
        ).rename("tidal_wetness"))
    )

# ── Full covariate stack builder (for AOI raster export) ─────────────────
def build_covariate_stack(region):
    """
    Build the full 27-band covariate stack for a region.
    Always uses _mask_s2_raster (cloud + tidal inundation) — correct for
    raster products; NOT for point extraction at tidal wetland cores.
    Calls _s2_raw_bands() and _s2_derived_bands() separately — matching
    the two-pass structure used in Cell 8 point extraction.
    Band order matches CANONICAL_BANDS exactly.
    """
    s2_composite = _s2_median_raster(region)
    ndvi_std = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
                .filterDate(S2_START, S2_END).filterBounds(region)
                .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", S2_MAX_CLOUD))
                .filter(ee.Filter.calendarRange(5, 9, "month"))
                .limit(S2_IMAGE_LIMIT, "CLOUDY_PIXEL_PERCENTAGE")
                .map(_mask_s2_raster)
                .map(lambda img: img.normalizedDifference(["B8","B4"]).rename("NDVI"))
                .reduce(ee.Reducer.stdDev()).rename("NDVI_stdDev"))
    return (stack_topo
            .addBands(stack_sar)
            .addBands(_s2_raw_bands(s2_composite))
            .addBands(_s2_derived_bands(s2_composite))
            .addBands(ndvi_std)
            .addBands(stack_climate)
            .toFloat().clip(region))

print("\n✅ Cell 5 done — all image stacks and S2 functions defined.")
print("   _s2_median_points() → _s2_raw_bands() + _s2_derived_bands()  : cloud mask only (Cell 8 point extraction)")
print("   _s2_median_raster() → _s2_raw_bands() + _s2_derived_bands()  : cloud + tidal mask (cov_raster, Cell 10 export)")
print("   build_covariate_stack(region)                                 : full 27-band AOI raster")


Building image stacks…
  ✓ Murray intertidal loaded
  ✓ Topography & Channels: ['elevation_m', 'slope', 'elevationRelMHW', 'twi', 'dist_to_channel_m', 'tidal_flat_prob', 'coastal_dist_m']
  ✓ S1: 942967 scenes loaded
  ✓ Sentinel-1 SAR: ['VV_mean', 'VH_mean', 'VVVH_ratio']
  ✓ TerraClimate: ['MAT_C', 'MAP_mm']
  ✓ Satellite Embeddings V1: 64 bands

✅ Cell 5 done — all image stacks and S2 functions defined.
   _s2_median_points() → _s2_raw_bands() + _s2_derived_bands()  : cloud mask only (Cell 8 point extraction)
   _s2_median_raster() → _s2_raw_bands() + _s2_derived_bands()  : cloud + tidal mask (cov_raster, Cell 10 export)
   build_covariate_stack(region)                                 : full 27-band AOI raster


In [13]:
# ──────────────────────────────────────────────────────────────────────────
# CELL 6 ── Visualise AOI covariates on map  (optional)
# ──────────────────────────────────────────────────────────────────────────

print("Building AOI composite for display (uses raster masking)…")
s2_viz    = _s2_median_raster(aoi_cov)
cov_check = (_s2_raw_bands(s2_viz)
             .addBands(_s2_derived_bands(s2_viz))
             .addBands(stack_topo).addBands(stack_sar).addBands(stack_climate)
             .toFloat().clip(aoi_cov))

MapViz = geemap.Map()
MapViz.add_basemap("SATELLITE")
MapViz.centerObject(aoi, zoom=10)
MapViz.addLayer(cov_check.select("NDVI_median"),
                {"min":-0.2,"max":0.8,"palette":["d73027","fee090","91cf60","1a9641"]},
                "NDVI median")
MapViz.addLayer(cov_check.select("tidal_wetness"),
                {"min":-0.35,"max":0.05,"palette":["ffffcc","41b6c4","0c2c84"]},
                "Tidal Wetness", False)
MapViz.addLayer(cov_check.select("VV_mean"),
                {"min":-25,"max":-5,"palette":["000000","cccccc","ffffff"]},
                "SAR VV", False)
MapViz.addLayer(cov_check.select("elevation_m"),
                {"min":-5,"max":10,"palette":["0571b0","92c5de","f7f7f7","d6604d","ca0020"]},
                "Elevation", False)
MapViz.addLayer(cov_check.select("dist_to_channel_m"),
                {"min":0,"max":2000,"palette":["08306b","6baed6","f7fbff"]},
                "Dist to Channel", False)
MapViz.addLayer(cov_check.select("mNDWI_median"),
                {"min":-0.5,"max":0.5,"palette":["a50026","ffffbf","4575b4"]},
                "mNDWI", False)
MapViz.addLayer(ee.FeatureCollection([ee.Feature(aoi)]),
                {"color":"FF0000","fillColor":"00000000"}, "AOI Boundary")
display(MapViz)
print("✅ Cell 6 done — toggle layers on the right.")


Building AOI composite for display (uses raster masking)…


Map(center=[48.81817524746727, -54.98299314175797], controls=(WidgetControl(options=['position', 'transparent_…

✅ Cell 6 done — toggle layers on the right.


In [14]:
# ──────────────────────────────────────────────────────────────────────────
# CELL 7 ── Extract TerraClimate + optional climate filter
# ──────────────────────────────────────────────────────────────────────────

# ── Diagnostic helper (used in Cells 7, 8, 9) ────────────────────────────
def _band_report(label, df, bands, warn_na=0.50, error_na=1.0):
    """
    Print NA rates and value ranges for `bands` columns in `df`.

    Parameters
    ----------
    label    : str   — printed header
    df       : DataFrame
    bands    : list  — column names to check
    warn_na  : float — flag bands with NA fraction above this (default 0.50)
    error_na : float — flag bands with NA fraction at or above this (default 1.0)

    Returns
    -------
    n_issues : int — number of bands at or above warn_na threshold
    """
    print(f"\n  ── {label} ──────────────────────────────────────────")
    n_issues = 0
    for b in bands:
        if b not in df.columns:
            print(f"    ✗ {b:<25} NOT IN DATAFRAME")
            n_issues += 1
            continue
        n   = len(df)
        na  = df[b].isna().sum()
        pct = na / n

        if pct >= error_na:
            sym = "✗"; n_issues += 1
            print(f"    {sym} {b:<25} NA=100%  — no data extracted")
        elif pct >= warn_na:
            sym = "⚠"; n_issues += 1
            print(f"    {sym} {b:<25} NA={pct*100:5.1f}%  "
                  f"[{df[b].min():.3g}, {df[b].max():.3g}]  mean={df[b].mean():.3g}  ← HIGH NA")
        else:
            sym = "✓"
            print(f"    {sym} {b:<25} NA={pct*100:5.1f}%  "
                  f"[{df[b].min():.3g}, {df[b].max():.3g}]  mean={df[b].mean():.3g}")

    if n_issues == 0:
        print(f"    All {len(bands)} bands OK")
    else:
        print(f"    ⚠ {n_issues}/{len(bands)} band(s) flagged")
    return n_issues

# ── Extraction helper functions ───────────────────────────────────────────
def _retry(image, batch, scale, tile_scale=2, attempt=0, max_attempts=4):
    """reduceRegions with recursive batch-halving retry on timeout."""
    if not batch: return [], 0
    fc = ee.FeatureCollection(batch)
    try:
        res = image.reduceRegions(collection=fc, reducer=ee.Reducer.first(),
                                   scale=scale, tileScale=tile_scale)
        return [f["properties"] for f in res.getInfo()["features"]], 0
    except Exception as e:
        err = str(e).lower()
        if (("timed out" in err or "computation" in err or "memory" in err)
                and len(batch) > 1 and attempt < max_attempts):
            mid = len(batch)//2
            r1,f1 = _retry(image, batch[:mid], scale, tile_scale, attempt+1, max_attempts)
            r2,f2 = _retry(image, batch[mid:], scale, tile_scale, attempt+1, max_attempts)
            return r1+r2, f1+f2
        return [], 1

def extract_static(image, features, label, batch_size=200, scale=30):
    """Extract a static global image at point locations."""
    results, n_fail = [], 0
    batches = [features[i:i+batch_size] for i in range(0, len(features), batch_size)]
    for b in tqdm(batches, desc=label):
        rows, fail = _retry(image, b, scale)
        results.extend(rows); n_fail += fail
    if n_fail:
        print(f"  ⚠ {label}: {n_fail} batch(es) permanently failed")
    return pd.DataFrame(results)

def extract_s2_raw(features, batch_size=25, scale=30):
    """
    Per-batch extraction of 9 raw S2 reflectance bands at point locations.

    Uses _s2_median_points() — cloud mask only, NO tidal inundation mask.
    Core profiles are in tidal marsh / seagrass by definition. The inundation
    mask removes valid pixels at these sites and returns 100% NA.

    Calls _s2_raw_bands() — same function used in build_covariate_stack() —
    guaranteeing identical band names across point and raster outputs.
    """
    results, n_fail = [], 0
    batches = [features[i:i+batch_size] for i in range(0, len(features), batch_size)]
    for b in tqdm(batches, desc="S2 raw bands"):
        fc     = ee.FeatureCollection(b)
        region = fc.geometry().bounds().buffer(S2_BUFFER_M)
        try:
            composite  = _s2_median_points(region)
            img        = _s2_raw_bands(composite)
            rows, fail = _retry(img, b, scale)
            results.extend(rows); n_fail += fail
        except Exception as e:
            n_fail += 1
            print(f"  ⚠ S2 raw batch failed: {e}")
    if n_fail:
        print(f"  ⚠ S2 raw bands: {n_fail} batch(es) permanently failed")
    return pd.DataFrame(results)

def extract_s2_derived(features, batch_size=25, scale=30):
    """
    Per-batch extraction of 5 derived S2 indices at point locations.

    Uses _s2_median_points() — cloud mask only, NO tidal inundation mask.
    Calls _s2_derived_bands() — same function used in build_covariate_stack().

    Kept as a separate pass from extract_s2_raw() so that a failure in derived
    index computation does not cause raw reflectance data to be lost, and to
    match the two-pass structure used in build_covariate_stack().
    """
    results, n_fail = [], 0
    batches = [features[i:i+batch_size] for i in range(0, len(features), batch_size)]
    for b in tqdm(batches, desc="S2 derived"):
        fc     = ee.FeatureCollection(b)
        region = fc.geometry().bounds().buffer(S2_BUFFER_M)
        try:
            composite  = _s2_median_points(region)
            img        = _s2_derived_bands(composite)
            rows, fail = _retry(img, b, scale)
            results.extend(rows); n_fail += fail
        except Exception as e:
            n_fail += 1
            print(f"  ⚠ S2 derived batch failed: {e}")
    if n_fail:
        print(f"  ⚠ S2 derived bands: {n_fail} batch(es) permanently failed")
    return pd.DataFrame(results)

def extract_ndvi_stddev(features, batch_size=25, scale=30):
    """
    Per-batch NDVI stdDev at point locations.
    Uses cloud mask only — same rationale as _s2_median_points() (no calendarRange).
    Must be per-batch — a globally pre-reduced stdDev image forces GEE
    to process the entire global S2 archive and reliably times out.
    """
    results, n_fail = [], 0
    batches = [features[i:i+batch_size] for i in range(0, len(features), batch_size)]
    for b in tqdm(batches, desc="NDVI stdDev"):
        fc     = ee.FeatureCollection(b)
        region = fc.geometry().bounds().buffer(S2_BUFFER_M)
        try:
            # No calendarRange filter — same rationale as _s2_median_points().
            col = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
                   .filterDate(S2_START, S2_END).filterBounds(region)
                   .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", S2_MAX_CLOUD))
                   .limit(S2_IMAGE_LIMIT, "CLOUDY_PIXEL_PERCENTAGE")
                   .map(_mask_s2_points)
                   .map(lambda img: img.normalizedDifference(["B8","B4"]).rename("NDVI")))
            img = col.reduce(ee.Reducer.stdDev()).rename("NDVI_stdDev").clip(region)
            rows, fail = _retry(img, b, scale)
            results.extend(rows); n_fail += fail
        except Exception as e:
            n_fail += 1
            print(f"  ⚠ NDVI stdDev batch: {e}")
    if n_fail:
        print(f"  ⚠ NDVI stdDev: {n_fail} batch(es) permanently failed")
    return pd.DataFrame(results)

def _make_features(df):
    return [ee.Feature(ee.Geometry.Point([float(r["longitude"]),float(r["latitude"])]),
                       {"profile_id":str(r["profile_id"]),"dataset":str(r["dataset"])})
            for _,r in df.iterrows()]

def _merge(main, sub, key="profile_id"):
    """
    Left-join extracted covariate columns onto the main profiles dataframe.
    Skips columns already present in main to prevent duplicate _x/_y suffixes
    (e.g. MAT_C and MAP_mm are pre-merged in Cell 7 and must not be re-merged).

    Guards against two failure modes:
    1. GEE returns feature properties that include a duplicate key column
       (e.g. profile_id appearing twice in sub.columns). pd.merge raises
       ValueError on non-unique column labels — deduplicate before selecting.
    2. sub is empty or has no new columns beyond the key — return main unchanged.
    """
    if sub is None or sub.empty: return main
    # Deduplicate columns in sub before any selection — GEE property dicts can
    # produce duplicate column names when the key is stored as both a geometry
    # property and a reducer output.
    sub = sub.loc[:, ~sub.columns.duplicated()]
    drop     = ["system:index", "dataset"]
    existing = set(main.columns)
    keep     = [key] + [c for c in sub.columns
                        if c not in drop and (c == key or c not in existing)]
    if len(keep) <= 1:
        return main
    return pd.merge(main, sub[keep], on=key, how="left")

# ── Build initial feature list ────────────────────────────────────────────
df_pts["profile_id"] = df_pts["profile_id"].astype(str)
features = _make_features(df_pts)
print(f"Feature list: {len(features):,} points")

# ── Extract TerraClimate first (fast — 4 km resolution) ──────────────────
print("\nExtracting TerraClimate (MAT/MAP)…")
df_climate = extract_static(stack_climate, features, "TerraClimate",
                             batch_size=500, scale=4000)
df_climate["profile_id"] = df_climate["profile_id"].astype(str)

_band_report("TerraClimate extraction", df_climate, ["MAT_C","MAP_mm"])

# ── Optional climate analog filter ───────────────────────────────────────
if USE_CLIMATE_FILTER:
    print("\nApplying climate analog filter…")
    aoi_samp  = (stack_climate.sample(region=aoi.centroid(maxError=100),
                                       scale=4000, numPixels=1, geometries=False)
                 .first().getInfo()["properties"])
    aoi_mat, aoi_map = aoi_samp["MAT_C"], aoi_samp["MAP_mm"]
    print(f"  AOI centroid: MAT = {aoi_mat:.1f}°C  |  MAP = {aoi_map:.0f} mm/yr")
    print(f"  Filter range: MAT {aoi_mat-MAT_TOLERANCE:.1f}–{aoi_mat+MAT_TOLERANCE:.1f}°C  |  "
          f"MAP {aoi_map*(1-MAP_TOLERANCE):.0f}–{aoi_map*(1+MAP_TOLERANCE):.0f} mm/yr")

    tmp  = pd.merge(df_pts, df_climate[["profile_id","MAT_C","MAP_mm"]],
                    on="profile_id", how="left")
    keep = ((tmp["MAT_C"].between(aoi_mat-MAT_TOLERANCE, aoi_mat+MAT_TOLERANCE) &
             tmp["MAP_mm"].between(aoi_map*(1-MAP_TOLERANCE), aoi_map*(1+MAP_TOLERANCE))) |
            tmp["MAT_C"].isna())
    df_pts  = tmp[keep].copy()
    removed = len(tmp) - len(df_pts)
    features = _make_features(df_pts)
    print(f"  Retained: {len(df_pts):,}  |  Removed: {removed:,}")

    if len(df_pts) < 30:
        print(f"  ⚠ Only {len(df_pts)} profiles remain after climate filter — "
              f"consider widening MAT_TOLERANCE / MAP_TOLERANCE")
else:
    # No filter — merge MAT_C/MAP_mm onto df_pts so Cell 8 _merge() skips them
    df_pts = pd.merge(df_pts, df_climate[["profile_id","MAT_C","MAP_mm"]],
                      on="profile_id", how="left")
    print("  Climate filter skipped — MAT_C/MAP_mm merged for output.")

print(f"\n✅ Cell 7 done — {len(features):,} profiles in working set.")



Feature list: 1,284 points

Extracting TerraClimate (MAT/MAP)…


TerraClimate:   0%|          | 0/3 [00:00<?, ?it/s]


  ── TerraClimate extraction ──────────────────────────────────────────
    ✓ MAT_C                     NA=  2.1%  [4.04, 28.5]  mean=14.2
    ✓ MAP_mm                    NA=  2.1%  [55.3, 3.55e+03]  mean=1.22e+03
    All 2 bands OK

Applying climate analog filter…
  AOI centroid: MAT = 4.9°C  |  MAP = 1225 mm/yr
  Filter range: MAT -3.1–12.9°C  |  MAP 490–1959 mm/yr
  Retained: 383  |  Removed: 901

✅ Cell 7 done — 383 profiles in working set.


In [16]:
# ── S2 DIAGNOSTIC — run in a new cell before Cell 8 ──────────────────────
# Checks the first profile point for: collection size, cloud stats,
# raw vs. masked pixel values, and whether the issue is the cloud
# filter, the QA60 mask, or something else entirely.

import pprint

# Use the first profile
test_row  = df_pts.iloc[0]
test_pt   = ee.Geometry.Point([float(test_row["longitude"]), float(test_row["latitude"])])
test_buf  = test_pt.buffer(S2_BUFFER_M)

print(f"Test point: profile_id={test_row['profile_id']}")
print(f"  lat={test_row['latitude']:.4f}  lon={test_row['longitude']:.4f}\n")

# 1. How many scenes exist with NO filters at all?
n_raw = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
         .filterDate(S2_START, S2_END)
         .filterBounds(test_buf)
         .size().getInfo())
print(f"[1] Scenes with date+bounds only           : {n_raw}")

# 2. How many survive the cloud filter?
n_cloud = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
           .filterDate(S2_START, S2_END)
           .filterBounds(test_buf)
           .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", S2_MAX_CLOUD))
           .size().getInfo())
print(f"[2] After CLOUDY_PIXEL_PERCENTAGE < {S2_MAX_CLOUD}%    : {n_cloud}")

# 3. How many survive with a relaxed cloud filter (80%)?
n_relaxed = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
             .filterDate(S2_START, S2_END)
             .filterBounds(test_buf)
             .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 80))
             .size().getInfo())
print(f"[3] After CLOUDY_PIXEL_PERCENTAGE < 80%    : {n_relaxed}")

# 4. Sample raw pixel value (NO cloud mask, NO divide) at the point
raw_val = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
           .filterDate(S2_START, S2_END)
           .filterBounds(test_buf)
           .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 80))
           .limit(5, "CLOUDY_PIXEL_PERCENTAGE")
           .median()
           .select(["B2","B3","B4","B8"])
           .reduceRegion(ee.Reducer.first(), test_pt, 30)
           .getInfo())
print(f"\n[4] Raw pixel values (no mask, no /10000)  : {raw_val}")
print(f"    (should be ~200–2000 for valid S2 SR pixels)")

# 5. Sample after _mask_s2_points (cloud mask + /10000)
masked_val = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
              .filterDate(S2_START, S2_END)
              .filterBounds(test_buf)
              .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 80))
              .limit(5, "CLOUDY_PIXEL_PERCENTAGE")
              .map(_mask_s2_points)
              .median()
              .select(["B2","B3","B4","B8"])
              .reduceRegion(ee.Reducer.first(), test_pt, 30)
              .getInfo())
print(f"\n[5] After _mask_s2_points (/10000)         : {masked_val}")
print(f"    (should be ~0.02–0.20 for valid surface reflectance)")

# 6. Check QA60 values directly — what bitmask values are present?
qa_vals = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
           .filterDate(S2_START, S2_END)
           .filterBounds(test_buf)
           .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 80))
           .limit(5, "CLOUDY_PIXEL_PERCENTAGE")
           .map(lambda img: img.select("QA60"))
           .median()
           .reduceRegion(ee.Reducer.first(), test_pt, 30)
           .getInfo())
print(f"\n[6] QA60 median value at point             : {qa_vals}")
print(f"    (0 = clear; 1024 = opaque cloud; 2048 = cirrus)")

Test point: profile_id=100
  lat=44.6254  lon=-124.0213

[1] Scenes with date+bounds only           : 298
[2] After CLOUDY_PIXEL_PERCENTAGE < 20%    : 83
[3] After CLOUDY_PIXEL_PERCENTAGE < 80%    : 192

[4] Raw pixel values (no mask, no /10000)  : {'B2': 261, 'B3': 286, 'B4': 149, 'B8': 24}
    (should be ~200–2000 for valid S2 SR pixels)

[5] After _mask_s2_points (/10000)         : {'B2': 0.026100000366568565, 'B3': 0.028599999845027924, 'B4': 0.01489999983459711, 'B8': 0.002400000113993883}
    (should be ~0.02–0.20 for valid surface reflectance)

[6] QA60 median value at point             : {'QA60': 0}
    (0 = clear; 1024 = opaque cloud; 2048 = cirrus)


In [15]:
# ──────────────────────────────────────────────────────────────────────────
# CELL 8 ── Extract all remote sensing stacks at core locations
#            Expect 20–60 min depending on dataset size.
# ──────────────────────────────────────────────────────────────────────────
# S2 extraction uses _s2_median_points() — cloud-mask-only composites —
# with _s2_raw_bands() and _s2_derived_bands() run as separate passes.
# The same two band functions are used by build_covariate_stack() for the
# raster export, guaranteeing identical band names and arithmetic across
# both paths.

print("Extracting remote sensing covariates at core locations…")
print("(batches auto-retry with smaller sizes on timeout)\n")

print("[1/5] Topography & Channels — 7 bands")
df_topo = extract_static(stack_topo, features, "Topography", batch_size=500, scale=30)
_band_report("Topography", df_topo,
             ["elevation_m","slope","elevationRelMHW","twi",
              "dist_to_channel_m","tidal_flat_prob","coastal_dist_m"])

print("\n[2/5] Sentinel-1 SAR — 3 bands")
df_sar = extract_static(stack_sar, features, "SAR", batch_size=100, scale=30)
_band_report("SAR", df_sar, ["VV_mean","VH_mean","VVVH_ratio"])

print("\n[3/5] Sentinel-2 raw reflectance — 9 bands  (cloud mask only, no inundation mask)")
df_s2r = extract_s2_raw(features, batch_size=25, scale=30)
_band_report("S2 raw reflectance", df_s2r, ["B","G","R","B5","B6","B7","NIR","SWIR1","SWIR2"])

# ── Plausibility checks on S2 reflectance ────────────────────────────────
# SR reflectance after /10000 should fall roughly in [0, 1].
for band in ["B","G","R","NIR","SWIR1","SWIR2"]:
    if band in df_s2r.columns:
        out_of_range = ((df_s2r[band] < 0) | (df_s2r[band] > 1)).sum()
        if out_of_range > 0:
            print(f"  ⚠ {band}: {out_of_range} values outside [0, 1] — check cloud masking")
# NDVI is a derived product; check it after the derived pass below

print("\n[4/5] Sentinel-2 derived indices — 5 bands  (cloud mask only, no inundation mask)")
df_s2d = extract_s2_derived(features, batch_size=25, scale=30)
_band_report("S2 derived indices", df_s2d,
             ["NDVI_median","LSWI_median","mNDWI_median","SAVI_median","tidal_wetness"])

if "NDVI_median" in df_s2d.columns:
    ndvi_bad = ((df_s2d["NDVI_median"] < -1) | (df_s2d["NDVI_median"] > 1)).sum()
    if ndvi_bad > 0:
        print(f"  ⚠ NDVI_median: {ndvi_bad} values outside [-1, 1] — check band arithmetic")

print("\n[5/5] NDVI standard deviation — 1 band  (cloud mask only)")
df_ndvi = extract_ndvi_stddev(features, batch_size=25, scale=30)
_band_report("NDVI stdDev", df_ndvi, ["NDVI_stdDev"])

# ── Merge everything ──────────────────────────────────────────────────────
# df_pts already carries MAT_C and MAP_mm from Cell 7.
# _merge() skips columns already present, preventing duplicate _x/_y suffixes.
print("\nMerging all extractions…")
df_out = df_pts.copy()
df_out["profile_id"] = df_out["profile_id"].astype(str)
for label, part in [("topo", df_topo), ("SAR", df_sar),
                     ("S2 raw", df_s2r), ("S2 derived", df_s2d),
                     ("NDVI stdDev", df_ndvi)]:
    before = set(df_out.columns)
    df_out = _merge(df_out, part)
    added  = set(df_out.columns) - before
    print(f"  + {label}: {len(added)} new columns added → {sorted(added)}")

# ── Enforce canonical column order; fill missing bands with NaN ───────────
n_missing = 0
for col in CANONICAL_BANDS:
    if col not in df_out.columns:
        print(f"  ⚠ Missing band after merge (extraction returned no data): {col}")
        df_out[col] = float("nan")
        n_missing += 1

if n_missing == 0:
    print(f"  ✓ All 27 canonical bands present after merge")
else:
    print(f"  ⚠ {n_missing} canonical band(s) missing — filled with NaN")

meta   = ["profile_id","dataset","latitude","longitude"]
extras = [c for c in df_out.columns if c not in meta + CANONICAL_BANDS]
df_out = df_out[meta + extras + CANONICAL_BANDS]

# ── Final coverage report ─────────────────────────────────────────────────
print(f"\n{'Band':<22} {'NA %':>6}  {'Min':>8}  {'Max':>8}  {'Mean':>8}")
print("─" * 62)
for b in CANONICAL_BANDS:
    pct  = df_out[b].isna().mean() * 100
    flag = "  ⚠ HIGH NA" if pct > 50 else ""
    flag = "  ✗ ALL NA" if pct >= 100 else flag
    if pct < 100:
        mn = f"{df_out[b].min():.3f}"
        mx = f"{df_out[b].max():.3f}"
        mu = f"{df_out[b].mean():.3f}"
    else:
        mn = mx = mu = "—"
    print(f"  {b:<20} {pct:>5.1f}%  {mn:>8}  {mx:>8}  {mu:>8}{flag}")

print(f"\n  Total: {len(df_out):,} rows × {len(df_out.columns)} cols")

# ── Build AOI covariate raster (needed for Cell 9 similarity) ─────────────
# Uses _mask_s2_raster (cloud + tidal inundation mask) — correct for raster
# products. Built here so it is available for build_similarity_raster() below.
print("\nBuilding AOI covariate raster (27 bands, raster masking)…")
cov_raster = build_covariate_stack(aoi_cov)

# Spot-check band names match canonical list
raster_bands = cov_raster.bandNames().getInfo()
raster_missing = [b for b in CANONICAL_BANDS if b not in raster_bands]
if raster_missing:
    print(f"  ⚠ Missing bands in cov_raster: {raster_missing}")
else:
    print(f"  ✓ All 27 canonical bands present in cov_raster")

print(f"\n✅ Cell 8 done — {len(df_out):,} rows × {len(df_out.columns)} cols")


Extracting remote sensing covariates at core locations…
(batches auto-retry with smaller sizes on timeout)

[1/5] Topography & Channels — 7 bands


Topography:   0%|          | 0/1 [00:00<?, ?it/s]


  ── Topography ──────────────────────────────────────────
    ✓ elevation_m               NA=  0.0%  [-3, 25]  mean=2.29
    ✓ slope                     NA=  0.0%  [0, 26.4]  mean=2.48
    ✓ elevationRelMHW           NA=  0.0%  [-3.5, 24.5]  mean=1.79
    ✓ twi                       NA=  0.0%  [0.698, 6.91]  mean=4.76
    ✓ dist_to_channel_m         NA=  0.0%  [0, 1.38e+03]  mean=110
    ✓ tidal_flat_prob           NA=  0.0%  [0, 1]  mean=0.337
    ✓ coastal_dist_m            NA=  0.0%  [0, 1.59e+03]  mean=158
    All 7 bands OK

[2/5] Sentinel-1 SAR — 3 bands


SAR:   0%|          | 0/4 [00:00<?, ?it/s]


  ── SAR ──────────────────────────────────────────
    ✓ VV_mean                   NA=  0.0%  [-23.1, -7.59]  mean=-14.4
    ✓ VH_mean                   NA=  0.0%  [-29.1, -13]  mean=-20.8
    ✓ VVVH_ratio                NA=  0.0%  [4.07, 11]  mean=6.34
    All 3 bands OK

[3/5] Sentinel-2 raw reflectance — 9 bands  (cloud mask only, no inundation mask)


S2 raw bands:   0%|          | 0/16 [00:00<?, ?it/s]


  ── S2 raw reflectance ──────────────────────────────────────────
    ✗ B                         NA=100%  — no data extracted
    ✗ G                         NA=100%  — no data extracted
    ✗ R                         NA=100%  — no data extracted
    ✗ B5                        NA=100%  — no data extracted
    ✗ B6                        NA=100%  — no data extracted
    ✗ B7                        NA=100%  — no data extracted
    ✗ NIR                       NA=100%  — no data extracted
    ✗ SWIR1                     NA=100%  — no data extracted
    ✗ SWIR2                     NA=100%  — no data extracted
    ⚠ 9/9 band(s) flagged

[4/5] Sentinel-2 derived indices — 5 bands  (cloud mask only, no inundation mask)


S2 derived:   0%|          | 0/16 [00:00<?, ?it/s]

KeyboardInterrupt: 